In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

True

In [2]:
from crewai_tools import SerperDevTool, ScrapeWebsiteTool

# 도구 인스턴스 생성
search_tool = SerperDevTool()
scrape_tool = ScrapeWebsiteTool(website_url='https://www.tripadvisor.co.kr/')

In [8]:
from typing import Type
from crewai.tools import BaseTool
from pydantic import BaseModel, Field

class CalculatorInput(BaseModel):
    expression: str = Field(
        ...,
        description="계산할 수식을 문자열로 입력하세요",
        json_schema_extra={"example": "2+2*3"}  # Pydantic v2 방식 예시
    )
    
class CalculatorTool(BaseTool):
    name: str = "calculator"
    description: str = "수학 계산을 수행합니다."
    args_schema: Type[BaseModel] = CalculatorInput

    def _run(self, expression: str) -> str:
        try:
            # eval 사용은 위험할 수 있으므로 안전한 eval 또는 parser 권장
            result = eval(expression, {"__builtins__": {}})
            return str(result)
        except Exception as e:
            return f"계산 오류: {e}"

In [9]:
# @tool 방식으로 tool 만들기
from crewai.tools import tool

@tool('calculator')
def calculator(expression: str) -> str:
    '''수학 계산을 수행합니다.'''
    try:
        result = eval(expression, {'__builtins__': {}})
        return str(result)
    except Exception as e:
        return f"계산 오류: {e}"

In [10]:
# 계산기 도구 인스턴스
calculator_tool = CalculatorTool()

In [11]:
from crewai import Agent, Task, Crew, Process

travel_agent = Agent(
    role="여행 전문가",
    goal="최적의 여행 일정과 예산 계획 제공",
    backstory="다년간의 여행 플래너 경험 보유",
    tools=[search_tool, scrape_tool, calculator_tool] ,
    llm="gpt-5.4-mini",
    verbose=True
)

travel_task = Task(
    description="{place} {days}일 여행 일정(문화, 미식 포함), 예산은 {budget}달러 (항공 {airfare}달러, 숙박 하루 {accommodation_per_day}달러).",
    expected_output="한국어로 작성된 {days}일간 {place} 여행에 대한 상세한 일정과 예산 계산 결과",
    agent=travel_agent
)

crew = Crew(
    agents=[travel_agent],
    tasks=[travel_task],
    process=Process.sequential, 
    verbose=True
)

In [12]:
result = await crew.kickoff_async(inputs={"place": "스페인", "days": 7, "budget": 5000, "airfare":1000, "accommodation_per_day":100})
print('[여행 시나리오 응답]\n', result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: a09e97b9-02fe-419a-9d5e-27a84de7b2f7                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: 스페인 7일 여행 일정(문화, 미식 포함), 예산은 5000달러 (항공 1000달러, 숙박 하루 100달러).               │
│  ID: 7871db8c-eefe-48ac-a019-b4cbe39bef4e                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 여행 전문가                                                                                             │
│                                                                                                                 │
│  Task: 스페인 7일 여행 일정(문화, 미식 포함), 예산은 5000달러 (항공 1000달러, 숙박 하루 100달러).               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 여행 전문가                                                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  좋습니다.                                                                                                      │
│  아래는 **스페인 7일 여행 일정(문화 + 미식 중심)**과 **예산 계산 결과**를 반영한 **실행 가능한 상세             │
│  플랜**입니다.                                                                                                  │
│  전제 조건은 다음과 같이 잡았습니다.                                                                            │
│                                                                                                                 │
│  - **총 예산:** 5,000달러                                                                                       │
│  - **항공권:** 1,000달러                                                                                        │
│  - **숙박비:** 하루 100달러 × 7박 = 700달러                                                                     │
│  - **남은 예산:** 3,300달러                                                                                     │
│  - 여행 스타일: **문화, 미식 중심 / 너무 빡빡하지 않게 / 도시 간 이동 포함**                                    │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  # 1) 전체 예산 계획                                                                                            │
│                                                                                                                 │
│  ## 총예산 요약                                                                                                 │
│  | 항목 | 금액 |                                                                                                │
│  |---|---:|                                                                                                     │
│  | 항공권 | $1,000 |                                                                                            │
│  | 숙박(7박) | $700 |                                                                                           │
│  | 식비 | $900 |                                                                                                │
│  | 도시 간 이동 | $300 |                                                                                        │
│  | 입장권/투어 | $450 |                                                                                         │
│  | 현지 교통 | $150 |                                                                                           │
│  | 예비비 | $500 |                                                                                              │
│  | **합계** | **$5,000** |                                                                                      │
│                                                                                                                 │
│  ### 예산 해설                                                                                                  │
│  - **숙박 7박 x $100 = $700**                                                                                   │
│  - **식비는 하루 평균 약 $128**                                                                                 │
│    - 점심은 타파스/메뉴 델 디아 활용                                                                            │
│    - 저녁은 

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 스페인 7일 여행 일정(문화, 미식 포함), 예산은 5000달러 (항공 1000달러, 숙박 하루 100달러).               │
│  Agent: 여행 전문가                                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: a09e97b9-02fe-419a-9d5e-27a84de7b2f7                                                                       │
│  Final Output: 좋습니다.                                                                                        │
│  아래는 **스페인 7일 여행 일정(문화 + 미식 중심)**과 **예산 계산 결과**를 반영한 **실행 가능한 상세             │
│  플랜**입니다.                                                                                                  │
│  전제 조건은 다음과 같이 잡았습니다.                                                                            │
│                                                                                                                 │
│  - **총 예산:** 5,000달러                                                                                       │
│  - **항공권:** 1,000달러                                                                                        │
│  - **숙박비:** 하루 100달러 × 7박 = 700달러                                                                     │
│  - **남은 예산:** 3,300달러                                                                                     │
│  - 여행 스타일: **문화, 미식 중심 / 너무 빡빡하지 않게 / 도시 간 이동 포함**                                    │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  # 1) 전체 예산 계획                                                                                            │
│                                                                                                                 │
│  ## 총예산 요약                                                                                                 │
│  | 항목 | 금액 |                                                                                                │
│  |---|---:|                                                                                                     │
│  | 항공권 | $1,000 |                                                                                            │
│  | 숙박(7박) | $700 |                                                                                           │
│  | 식비 | $900 |                                                                                                │
│  | 도시 간 이동 | $300 |                                                                                        │
│  | 입장권/투어 | $450 |                                                                                         │
│  | 현지 교통 | $150 |                                                                                           │
│  | 예비비 | $500 |                                                                                              │
│  | **합계** | **$5,000** |                                                                                      │
│                                                                                                                 │
│  ### 예산 해설                                                                                                  │
│  - **숙박 7박 x $100 = $700**                                                                                   │
│  - **식비는 하루 평균 약 $128**                                                                                 │
│    - 점심은 타파스/메뉴 델 디아 활용                                                                            │
│    

[여행 시나리오 응답]
 좋습니다.  
아래는 **스페인 7일 여행 일정(문화 + 미식 중심)**과 **예산 계산 결과**를 반영한 **실행 가능한 상세 플랜**입니다.  
전제 조건은 다음과 같이 잡았습니다.

- **총 예산:** 5,000달러
- **항공권:** 1,000달러
- **숙박비:** 하루 100달러 × 7박 = 700달러
- **남은 예산:** 3,300달러
- 여행 스타일: **문화, 미식 중심 / 너무 빡빡하지 않게 / 도시 간 이동 포함**

---

# 1) 전체 예산 계획

## 총예산 요약
| 항목 | 금액 |
|---|---:|
| 항공권 | $1,000 |
| 숙박(7박) | $700 |
| 식비 | $900 |
| 도시 간 이동 | $300 |
| 입장권/투어 | $450 |
| 현지 교통 | $150 |
| 예비비 | $500 |
| **합계** | **$5,000** |

### 예산 해설
- **숙박 7박 x $100 = $700**
- **식비는 하루 평균 약 $128**
  - 점심은 타파스/메뉴 델 디아 활용
  - 저녁은 1~2회는 좋은 레스토랑 포함
- **도시 간 이동**
  - 바르셀로나 → 마드리드: 고속열차(AVE) 또는 항공
  - 마드리드 → 세비야: 고속열차(AVE)
- **예비비 $500**
  - 쇼핑, 추가 식사, 택시, 소소한 업그레이드 대응용

---

# 2) 추천 여행 루트

가장 효율적인 7일 코스는 아래입니다.

**바르셀로나 3일 → 마드리드 2일 → 세비야 2일**

이 루트가 좋은 이유:
- 스페인의 대표 문화권을 균형 있게 경험 가능
- 예술/건축/미식/플라멩코가 모두 포함됨
- 고속열차 이동이 편리하고 동선이 자연스러움

---

# 3) 7일 상세 일정

---

## DAY 1. 바르셀로나 도착 - 고딕지구 & 타파스
### 핵심 테마
- 첫날은 시차 적응 + 도심 산책 + 스페인식 저녁

### 일정
- **오전/오후**
  - 바르셀로나 도착
  - 호텔 체크인 후 휴

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯